In [4]:
from pathlib import Path
import warnings
import json
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

GLOBAL_CONFIG = {
    "datasets": [
        {"name": "M", "input_path": "m_data.csv", "output_subdir": "M"},
        {"name": "W", "input_path": "w_data.csv", "output_subdir": "W"},
    ],
    "output_root": "season_splits",
    "season_col": "Season",
    "day_col_candidates": ["DayNum", "GameDayNum", "RankingDayNum"],
    "date_col_candidates": ["GameDate", "Date", "game_date", "date"],
    "game_id_col_candidates": ["GameID", "game_id", "ID", "MatchID", "match_id", "meczid", "MeczID"],
    "valid_seasons": 1,
    "test_seasons": 1,
    "min_train_floor": 3,
    "save_index": False,
}

OUTPUT_ROOT = Path(GLOBAL_CONFIG["output_root"])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Katalog roboczy:", Path.cwd())
print("Folder wyjściowy:", OUTPUT_ROOT.resolve())
print("Pliki wejściowe:")
for ds in GLOBAL_CONFIG["datasets"]:
    p = Path(ds["input_path"])
    print(f"- {ds['name']}: {p} | exists={p.exists()}")


def first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


def infer_columns(df, config):
    season_col = config["season_col"]
    if season_col not in df.columns:
        raise ValueError(
            f"Brakuje kolumny `{season_col}`. "
            f"Dostępne kolumny: {list(df.columns)[:30]}..."
        )

    day_col = first_existing(df.columns, config["day_col_candidates"])
    date_col = first_existing(df.columns, config["date_col_candidates"])
    game_id_col = first_existing(df.columns, config["game_id_col_candidates"])

    return season_col, day_col, date_col, game_id_col


def prepare_chronology(df, season_col, day_col=None, date_col=None, game_id_col=None):
    out = df.copy()
    out[season_col] = pd.to_numeric(out[season_col], errors="raise").astype(int)

    if day_col is not None:
        out[day_col] = pd.to_numeric(out[day_col], errors="coerce")
    if date_col is not None:
        out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

    if day_col is None and date_col is None:
        warnings.warn(
            "Nie znaleziono ani DayNum, ani Date/GameDate. Split będzie robiony tylko po kolumnie Season."
        )

    out["_row_order"] = np.arange(len(out))

    sort_cols = [season_col]
    if day_col is not None:
        sort_cols.append(day_col)
    elif date_col is not None:
        if out[date_col].isna().all():
            raise ValueError(
                f"Kolumna `{date_col}` istnieje, ale po parsowaniu wszystkie daty są NaT."
            )
        sort_cols.append(date_col)

    if game_id_col is not None:
        sort_cols.append(game_id_col)

    sort_cols.append("_row_order")
    out = out.sort_values(sort_cols).reset_index(drop=True)
    return out, sort_cols


def safe_basic_eda(df, dataset_name, season_col, day_col=None, date_col=None, game_id_col=None, max_rows=20):
    print(f"===== BASIC SAFE EDA [{dataset_name}] =====")
    print("Liczba wierszy:", len(df))
    print("Liczba kolumn:", df.shape[1])

    print("\nLiczba meczów per sezon:")
    season_counts = df.groupby(season_col).size().rename("n_games").reset_index()
    display(season_counts)

    if day_col is not None:
        print("\nZakres DayNum per sezon:")
        day_summary = df.groupby(season_col)[day_col].agg(["min", "max", "nunique"]).reset_index()
        display(day_summary.head(max_rows))

    if date_col is not None:
        print("\nZakres dat per sezon:")
        date_summary = df.groupby(season_col)[date_col].agg(["min", "max"]).reset_index()
        display(date_summary.head(max_rows))

    if game_id_col is not None:
        dupes = df[game_id_col].duplicated().sum()
        print(f"\nDuplikaty po `{game_id_col}`:", dupes)
        if dupes > 0:
            warnings.warn(
                f"[{dataset_name}] Masz {dupes} duplikatów po `{game_id_col}`. "
                "Jeśli 1 wiersz ma oznaczać 1 mecz, trzeba to sprawdzić przed trenowaniem."
            )


def find_potential_leakage_columns(columns):
    suspicious_tokens = [
        "score", "result", "winner", "loser", "margin", "mov", "ot",
        "fgm", "fga", "fgm3", "fga3", "ftm", "fta",
        "reb", "or", "dr", "ast", "to", "stl", "blk", "pf",
        "wscore", "lscore", "wloc", "numot"
    ]
    found = []
    for col in columns:
        c = str(col).lower()
        if any(tok in c for tok in suspicious_tokens):
            found.append(col)
    return sorted(set(found))


def resolve_split_params(unique_seasons, config, dataset_name):
    n = len(unique_seasons)
    valid_seasons = int(config["valid_seasons"])
    test_seasons = int(config["test_seasons"])
    min_train_floor = int(config.get("min_train_floor", 1))

    required = valid_seasons + test_seasons + min_train_floor
    if n < required:
        raise ValueError(
            f"[{dataset_name}] Za mało sezonów ({n}) na taki split. "
            f"Potrzeba co najmniej {required}: "
            f"{min_train_floor} train + {test_seasons} test + {valid_seasons} valid."
        )

    train_seasons_count = n - valid_seasons - test_seasons

    return {
        "train_seasons_count": train_seasons_count,
        "test_seasons": test_seasons,
        "valid_seasons": valid_seasons,
        "notes": [],
    }


def summarize_frame(frame, dataset_name, split_name, role, path, season_col, day_col=None, date_col=None):
    row = {
        "dataset": dataset_name,
        "split_name": split_name,
        "role": role,
        "path": str(path),
        "rows": len(frame),
        "first_season": int(frame[season_col].min()) if len(frame) else np.nan,
        "last_season": int(frame[season_col].max()) if len(frame) else np.nan,
    }
    if day_col is not None and len(frame):
        row["min_daynum"] = frame[day_col].min()
        row["max_daynum"] = frame[day_col].max()
    else:
        row["min_daynum"] = np.nan
        row["max_daynum"] = np.nan

    if date_col is not None and len(frame):
        row["min_date"] = frame[date_col].min()
        row["max_date"] = frame[date_col].max()
    else:
        row["min_date"] = pd.NaT
        row["max_date"] = pd.NaT
    return row


def to_jsonable(obj):
    if isinstance(obj, (pd.Timestamp, np.datetime64)):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Nie da się zserializować typu: {type(obj)}")


def generate_split_for_dataset(dataset_cfg, global_config, show_eda=True):
    dataset_name = dataset_cfg["name"]
    input_path = Path(dataset_cfg["input_path"])
    output_dir = OUTPUT_ROOT / dataset_cfg["output_subdir"]
    output_dir.mkdir(parents=True, exist_ok=True)

    if not input_path.exists():
        warnings.warn(f"[{dataset_name}] Pomijam, bo nie znaleziono pliku: {input_path.resolve()}")
        return None

    print("\n" + "=" * 90)
    print(f"DATASET: {dataset_name} | plik: {input_path}")
    print("=" * 90)

    df = pd.read_csv(input_path)
    print("Shape:", df.shape)
    display(df.head())

    season_col, day_col, date_col, game_id_col = infer_columns(df, global_config)
    df, sort_cols = prepare_chronology(
        df=df,
        season_col=season_col,
        day_col=day_col,
        date_col=date_col,
        game_id_col=game_id_col,
    )

    print("Użyte kolumny chronologiczne:")
    print({
        "dataset": dataset_name,
        "season_col": season_col,
        "day_col": day_col,
        "date_col": date_col,
        "game_id_col": game_id_col,
        "sort_cols": sort_cols,
    })

    if show_eda:
        safe_basic_eda(df, dataset_name, season_col, day_col, date_col, game_id_col)

    suspicious = find_potential_leakage_columns(df.columns)
    print("\nPotencjalnie podejrzane kolumny (do audytu przed modelowaniem):")
    print(suspicious[:100])

    unique_seasons = sorted(df[season_col].dropna().unique().tolist())
    print("\nSezony:", unique_seasons)

    split_params = resolve_split_params(unique_seasons, global_config, dataset_name)
    print("\nResolved split params:")
    print(split_params)

    valid_n = split_params["valid_seasons"]
    test_n = split_params["test_seasons"]

    # TU JEST ZMIANA:
    # validation = ostatni sezon(y)
    # test = przedostatni sezon(y)
    # train = cała wcześniejsza historia
    valid_seasons_list = unique_seasons[-valid_n:]
    test_seasons_list = unique_seasons[-(valid_n + test_n):-valid_n]
    train_seasons_list = unique_seasons[:-(valid_n + test_n)]

    train_df = df[df[season_col].isin(train_seasons_list)].copy()
    test_df = df[df[season_col].isin(test_seasons_list)].copy()
    valid_df = df[df[season_col].isin(valid_seasons_list)].copy()

    train_path = output_dir / "train.csv"
    test_path = output_dir / "test.csv"
    valid_path = output_dir / "valid.csv"

    train_df.to_csv(train_path, index=global_config["save_index"])
    test_df.to_csv(test_path, index=global_config["save_index"])
    valid_df.to_csv(valid_path, index=global_config["save_index"])

    manifest_rows = [
        summarize_frame(train_df, dataset_name, "season_split", "train", train_path, season_col, day_col, date_col),
        summarize_frame(test_df, dataset_name, "season_split", "test", test_path, season_col, day_col, date_col),
        summarize_frame(valid_df, dataset_name, "season_split", "valid", valid_path, season_col, day_col, date_col),
    ]

    manifest = pd.DataFrame(manifest_rows)
    manifest_path = output_dir / "split_manifest.csv"
    manifest.to_csv(manifest_path, index=False)

    dataset_config = {
        "dataset": dataset_name,
        "input_path": str(input_path),
        "output_dir": str(output_dir),
        "season_col": season_col,
        "day_col": day_col,
        "date_col": date_col,
        "game_id_col": game_id_col,
        "sort_cols": sort_cols,
        "resolved_split_params": split_params,
        "train_seasons": train_seasons_list,
        "test_seasons": test_seasons_list,
        "valid_seasons": valid_seasons_list,
    }

    config_path = output_dir / "split_config.json"
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(dataset_config, f, ensure_ascii=False, indent=2, default=to_jsonable)

    print("\nZapisano split:")
    print("train:", train_path.resolve())
    print("test :", test_path.resolve())
    print("valid:", valid_path.resolve())

    print("\nZakres sezonów:")
    print("train:", train_seasons_list[0], "->", train_seasons_list[-1], "|", train_seasons_list)
    print("test :", test_seasons_list)
    print("valid:", valid_seasons_list)

    print("\nShapes:")
    print("train:", train_df.shape)
    print("test :", test_df.shape)
    print("valid:", valid_df.shape)

    print("\nManifest:")
    display(manifest)

    return {
        "dataset": dataset_name,
        "input_path": input_path,
        "output_dir": output_dir,
        "manifest": manifest,
        "manifest_path": manifest_path,
        "config_path": config_path,
        "resolved_split_params": split_params,
    }


results = []
for dataset_cfg in GLOBAL_CONFIG["datasets"]:
    result = generate_split_for_dataset(dataset_cfg, GLOBAL_CONFIG, show_eda=True)
    if result is not None:
        results.append(result)

if not results:
    raise FileNotFoundError(
        "Nie znaleziono żadnego z plików wejściowych (`m_data.csv`, `w_data.csv`). "
        "Umieść je obok notebooka albo popraw ścieżki w GLOBAL_CONFIG['datasets']."
    )

all_manifests = pd.concat([r["manifest"] for r in results], ignore_index=True)
all_manifest_path = OUTPUT_ROOT / "split_manifest_all.csv"
all_manifests.to_csv(all_manifest_path, index=False)

run_summary = pd.DataFrame([
    {
        "dataset": r["dataset"],
        "input_path": str(r["input_path"]),
        "output_dir": str(r["output_dir"]),
        "notes": " | ".join(r["resolved_split_params"].get("notes", [])),
    }
    for r in results
])
run_summary_path = OUTPUT_ROOT / "split_run_summary.csv"
run_summary.to_csv(run_summary_path, index=False)

print("\nZbiorczy manifest:", all_manifest_path.resolve())
print("Podsumowanie runu:", run_summary_path.resolve())
display(run_summary)

print("Folder z wynikami:", OUTPUT_ROOT.resolve())
print("Pliki końcowe:")
for p in sorted(OUTPUT_ROOT.rglob("*.csv")):
    print("-", p)

Katalog roboczy: c:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026
Folder wyjściowy: C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\season_splits
Pliki wejściowe:
- M: m_data.csv | exists=True
- W: w_data.csv | exists=True

DATASET: M | plik: m_data.csv
Shape: (125480, 93)


C:\Users\Stefan\AppData\Local\Temp\ipykernel_25568\1732440175.py:214: DtypeWarning: Columns (21,23,24,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


,GameID,Gender,Season,DayNum,GameType,isRegularSeason,isTourney,HasBoxscore,WTeamID,LTeamID,WScore,LScore,ScoreDiff,TotalScore,WLoc,NeutralSite,WHome,WAway,LHome,LAway,NumOT,WSeed,WSeedNum,WSeedRegion,LSeed,LSeedNum,LSeedRegion,WCoachName,LCoachName,WMasseyMeanRank,WMasseyMedianRank,WMasseyBestRank,WMasseyWorstRank,LMasseyMeanRank,LMasseyMedianRank,LMasseyBestRank,LMasseyWorstRank,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,WCoachStartDayNum,WCoachEndDayNum,LCoachStartDayNum,LCoachEndDayNum,WMasseyDayNum,WMasseySystemCount,WMasseyTop10Count,WMasseyTop25Count,WMassey_BOB,WMassey_COL,WMassey_DOK,WMassey_MAS,WMassey_MOR,WMassey_POM,WMassey_RPI,WMassey_SAG,WMassey_WLK,LMasseyDayNum,LMasseySystemCount,LMasseyTop10Count,LMasseyTop25Count,LMassey_BOB,LMassey_COL,LMassey_DOK,LMassey_MAS,LMassey_MOR,LMassey_POM,LMassey_RPI,LMassey_SAG,LMassey_WLK
0,M_2003_10_1104_1328,M,2003,10,RegularSeason,1,0,1,1104,1328,68,62,6,130,N,1,0,0,0,0,0,Y10,10.0,Y,W01,1.0,W,mark_gottfried,kelvin_sampson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20,0.0,154.0,0.0,154.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M_2003_10_1272_1393,M,2003,10,RegularSeason,1,0,1,1272,1393,70,63,7,133,N,1,0,0,0,0,0,Z07,7.0,Z,W03,3.0,W,john_calipari,jim_boeheim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16,0.0,154.0,0.0,154.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M_2003_11_1186_1458,M,2003,11,RegularSeason,1,0,1,1458,1186,81,55,26,136,H,0,1,0,0,1,0,Y05,5.0,Y,NaN,NaN,NaN,bo_ryan,ray_giacoletti,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26,57,6,12,23,27,12,24,12,9,9,3,18,20,46,3,11,12,17,6,22,8,19,4,3,25,0.0,154.0,0.0,154.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M_2003_11_1208_1400,M,2003,11,RegularSeason,1,0,1,1400,1208,77,71,6,148,N,1,0,0,0,0,0,X01,1.0,X,NaN,NaN,NaN,rick_barnes,jim_harrick,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30,61,6,14,11,13,17,22,12,14,4,4,20,24,62,6,16,17,27,21,15,12,10,7,1,14,0.0,154.0,0.0,154.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M_2003_11_1266_1437,M,2003,11,RegularSeason,1,0,1,1266,1437,73,61,12,134,N,1,0,0,0,0,0,Y03,3.0,Y,NaN,NaN,NaN,tom_crean,jay_wright,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,58,8,18,17,29,17,26,15,10,5,2,25,22,73,3,26,14,23,31,22,9,12,2,5,23,0.0,154.0,0.0,154.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Użyte kolumny chronologiczne:
{'dataset': 'M', 'season_col': 'Season', 'day_col': 'DayNum', 'date_col': None, 'game_id_col': 'GameID', 'sort_cols': ['Season', 'DayNum', 'GameID', '_row_order']}
===== BASIC SAFE EDA [M] =====
Liczba wierszy: 125480
Liczba kolumn: 94

Liczba meczów per sezon:


,Season,n_games
0,2003,4680
1,2004,4635
2,2005,4739
3,2006,4821
4,2007,5107
5,2008,5227
6,2009,5313
7,2010,5327
8,2011,5313
9,2012,5320


...


Duplikaty po `GameID`: 0

Potencjalnie podejrzane kolumny (do audytu przed modelowaniem):
['HasBoxscore', 'LAst', 'LBlk', 'LDR', 'LFGA', 'LFGA3', 'LFGM', 'LFGM3', 'LFTA', 'LFTM', 'LOR', 'LPF', 'LScore', 'LSeedRegion', 'LStl', 'LTO', 'NumOT', 'ScoreDiff', 'TotalScore', 'WAst', 'WBlk', 'WDR', 'WFGA', 'WFGA3', 'WFGM', 'WFGM3', 'WFTA', 'WFTM', 'WLoc', 'WOR', 'WPF', 'WScore', 'WSeedRegion', 'WStl', 'WTO', '_row_order', 'isTourney']

Sezony: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Resolved split params:
{'train_seasons_count': 15, 'test_seasons': 1, 'valid_seasons': 1, 'notes': []}

Zapisano split:
train: C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\season_splits\W\train.csv
test : C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\season_splits\W\test.csv
valid: C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\season_splits\W\valid.csv

Zakres sezonów:
train: 2010 -> 2024 | [2010, 2011, 2012, 2013,

,dataset,split_name,role,path,rows,first_season,last_season,min_daynum,max_daynum,min_date,max_date
0,W,season_split,train,season_splits\W\train.csv,77158,2010,2024,0,155,NaT,NaT
1,W,season_split,test,season_splits\W\test.csv,5511,2025,2025,0,153,NaT,NaT
2,W,season_split,valid,season_splits\W\valid.csv,5065,2026,2026,0,118,NaT,NaT



Zbiorczy manifest: C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\season_splits\split_manifest_all.csv
Podsumowanie runu: C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\season_splits\split_run_summary.csv


,dataset,input_path,output_dir,notes
0,M,m_data.csv,season_splits\M,
1,W,w_data.csv,season_splits\W,


Folder z wynikami: C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\season_splits
Pliki końcowe:
- season_splits\M\split_manifest.csv
- season_splits\M\test.csv
- season_splits\M\train.csv
- season_splits\M\valid.csv
- season_splits\split_manifest_all.csv
- season_splits\split_run_summary.csv
- season_splits\W\split_manifest.csv
- season_splits\W\test.csv
- season_splits\W\train.csv
- season_splits\W\valid.csv
